# RTENSO2026: sudden stratospheric warmings and the polar vortex, 2026 starts

The target is the initialized RTENSO2026 forecasts themselves: what they say about the vortex, how
that depends on the start and the initial-state setup, and how realistic their simulated SSWs are. The
notebook answers, in order:

1. **Feb 2026 start** (verification): how the ensemble represents the rest of the observed 2025/26
   winter, the late-February 2026 disruption and the spring final warming.
2. **Winter 2026/27**: the vortex state and SSW likelihood the forecasts favour, in the ENSO and QBO
   state they simulate.
3. **Winter 2027/28**: the same at longer lead (May and Aug starts; the Feb start ends 31 Jan 2028).
4. **Sensitivity**: differences by start, by case (`clim`, `obs`, `obs_ice_nsidc`) and across members,
   and to the drift correction.
5. **Event dynamics**: whether simulated reversals have realistic wave forcing, vertical structure,
   persistence and recovery.

Two forecast winters from one initial state are not a model climatology: SSW counts are conditional
forecast statistics, the ERA5 record is context, and no model SSW-frequency bias is inferred from them.

The Northern Hemisphere polar vortex in the E3SM RTENSO2026 hindcasts from the 1 Feb, 1 May and
1 Aug 2026 starts, against ERA5. This covers the end of winter 2025/26 (Feb start only) and the
forecast winters 2026/27 (El Niño) and 2027/28.

* **Model climatology** (section 3): the seasonal cycle of the 10-hPa vortex wind and the
  winter zonal-mean wind in the Reanalysis (BruteForce) 1981-2010 hindcasts against ERA5.
  This bias affects how many SSWs the forecasts can produce.
* **QBO** (section 4): equatorial zonal wind from each start against ERA5, and how long
  the initial QBO phase persists.
* **Winter 2025/26** (section 5): the observed vortex and final warming, and the 1 Feb 2026 start.
* **2026/27 and 2027/28 forecasts** (section 6): daily U(60°N, 10 hPa) plumes, SSW counts per
  member-winter and final-warming dates against ERA5 1979-2026, and tercile probabilities of
  the DJF vortex strength.
* **ENSO and QBO across members** (section 7): member DJF vortex against member Niño3.4 and
  autumn QBO, next to the observed winters.
* **SSW composites** (section 8): U(60°N, 10 hPa), the 100-hPa eddy heat flux and the polar-cap
  height ("dripping paint") around the central dates, pooled across all model SSWs, against ERA5 1979-2025.

**Data.** Model daily values are the native `eam.h1` daily means (U/V/T at 22 levels up to 1 hPa,
Z at 16 levels up to 10 hPa), which have not been post-processed. They are averaged into 1° latitude
bands (`util/strat_case.py`) and cached under `$S2D_DIAG_ROOT/<source>/strat/`. Monthly values
come from the post-processed 1x1 U, T, Z3 (19 levels). ERA5 is the daily zonal-mean archive
`ERA5/1x1/daily_zonal` (zonal means of the 00Z and 12Z analyses on 37 levels, built by
`ERA5/scripts/daily_zonal`). It runs to the last complete month on the NERSC mirror, so the 2026/27
winter is not yet observed.

**Definitions.** SSW: the daily zonal-mean U at 60°N (the 59.5/60.5°N bands), 10 hPa, turns
easterly between 1 Nov and 31 Mar. Events must be separated by 20 consecutive westerly days, and the
wind must return to westerly for 10 consecutive days before 30 Apr (Charlton and Polvani 2007; Butler
et al. 2017). Final warming: the spring onset of easterlies after which the wind does not stay
westerly for 10 days before 31 Jul. Model SSWs are counted on the raw model wind
and on a **bias-adjusted** wind (raw minus the difference between the model and ERA5 climatologies
on that day), because a too-strong or too-weak model vortex changes how often it reverses.

**Anomalies.** Model anomalies remove the Reanalysis 1981-2010 lead-time climatology
(May/Nov starts; Feb/Aug use `CLIM_METHOD = "lead_drift"`: the ERA5 target-month climatology plus
the mean May/Nov drift at the same lead). Daily model anomalies use that monthly climatology
interpolated to days (for Z plus each run's ensemble-mean offset between the native daily Z and the
monthly Z3 product, ~40 m at 10 hPa: `sc.daily_monthly_offset`), and daily ERA5 anomalies use a smoothed 1981-2010 daily climatology
(4 harmonics). Polar-cap height anomalies are divided by the ERA5 daily standard deviation for both.

**Pooled ensembles:** each start pools the `POOL` cases (30 members). The 1 Feb 2026 `clim` and `obs`
runs are bit-identical (see "1 Feb 2026 start" in `README.md`), so that start pools 20 members.

In [ ]:
%matplotlib inline

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.lines import Line2D

from util import rtenso_case as rc
from util import strat_case as sc

warnings.filterwarnings("ignore", message="invalid value encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="All-NaN slice")

### Run controls

In [ ]:
NOTEBOOK = "9_ssw_polar_vortex"   # outputs: rc.OUTPUT_ROOT/9_ssw_polar_vortex/{figures,data}/
# "auto": reuse the zonal-mean caches; "rebuild": re-read the model output; "require": fail if a cache is missing
derivation_mode = "auto"
CLIM_METHOD = "lead_drift"        # Feb/Aug drift correction: "lead_drift" or "target_month"
WORKERS = 4                       # processes reading model output (cache builds only; login node: 30 GB per user)

# Data locations (machine-specific)
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")                       # reference-data archive
ERA5_ZONAL_DIR = OBSERVATION_ROOT / "ERA5" / "1x1" / "daily_zonal"                    # ERA5 daily zonal means
ERA5_DAILY_DIR = OBSERVATION_ROOT / "ERA5" / "1x1" / "daily"                          # ERA5 daily maps (PSL)
SIMULATION_ROOT = Path("/global/cfs/cdirs/e3sm/S2S2D/RTENSO2026/simulation")         # raw eam.h1 history

YEAR = 2026
INIT_MONTHS = (2, 5, 8)
MONTH_NAMES = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
               9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
INIT_COLORS = {2: "#1f77b4", 5: "#2ca02c", 8: "#d62728"}
MODEL_PREFIX = rc.MODEL_PREFIX
POOL = {
    "Reanalysis-CLIM": f"{MODEL_PREFIX}_RTENSO2026_clim",
    "Reanalysis-OBS": f"{MODEL_PREFIX}_RTENSO2026_obs",
    "Reanalysis-NSIDC": f"{MODEL_PREFIX}_RTENSO2026_obs_ice_nsidc",
}

CLIM_YEARS = rc.CLIM_YEARS          # 1981-2010: model drift and ERA5 anomaly climatology
ERA5_FIRST_YEAR = 1979              # ERA5 daily record read (SSW climatology from winter 1979/80)
REF_LEVEL = 10.0                    # hPa, SSW wind
QBO_LEVEL = 50.0                    # hPa, QBO phase (Holton-Tan)
QBO_MONTHS = (10, 11)               # QBO phase months before the winter
DJF = ((0, 12), (1, 1), (1, 2))     # (years after the winter year, month)
SSW = {"sep_days": 20, "recover_days": 10}
NINO_PRODUCT_OBS = "HadISST1.1"     # observed DJF Nino3.4 (1870 on)
ENSO_THRESHOLD = 0.5                # degC: El Nino / La Nina winters (DJF Nino3.4)
LAGS = np.arange(-40, 61)           # days around SSW central dates
PCH_LAGS = (-30, 60)

fontz = 13
fig_type = "png"
fig_dpi = 200
plt.rcParams.update({"font.size": fontz, "axes.titlesize": fontz, "axes.labelsize": fontz,
                     "legend.fontsize": fontz * 0.8, "xtick.labelsize": fontz * 0.85, "ytick.labelsize": fontz * 0.85})


def save(fig, *parts):
    path = rc.save_figure(fig, NOTEBOOK, *parts, dpi=fig_dpi, ext=fig_type)
    print(f"[FIGURE] {path}")


DATA_DIR = rc.output_dir(NOTEBOOK, "data")

### 1. ERA5

Daily zonal means from `ERA5_FIRST_YEAR` to the last complete month, the smoothed 1981-2010 daily
climatology and standard deviation of each index, and calendar-month means for the QBO and the
monthly drift climatology.

In [ ]:
%%time
era5 = sc.era5_daily_zonal(ERA5_ZONAL_DIR, years=(ERA5_FIRST_YEAR, YEAR), load=False)   # lazy: only reductions are loaded
obs_end = pd.Timestamp(era5["time"].values[-1])
print(f"ERA5 daily zonal means {pd.Timestamp(era5['time'].values[0]):%d %b %Y} - {obs_end:%d %b %Y}")


def doy_of(da):
    return xr.DataArray(sc.noleap_doy(da["time"].values), dims="time", coords={"time": da["time"]})


def on_days(clim, da):
    # a (doy, ...) climatology on the days of da
    return clim.sel(doy=doy_of(da)).drop_vars("doy")


# indices
e_u60 = sc.lat_mean(era5["U"].sel(plev=REF_LEVEL), sc.U60).load()       # (time)
e_pcz = sc.lat_mean(era5["Z"], sc.PCAP).load()                           # (time, plev)
e_pct = sc.lat_mean(era5["T"].sel(plev=REF_LEVEL), sc.PCAP).load()
e_vt = sc.lat_mean(era5["VT"].sel(plev=100.0), sc.VT_BAND).load()
e_qbo = sc.monthly_means(sc.lat_mean(era5["U"], sc.EQ)).load()           # (time, plev), monthly

clim_u60, std_u60 = sc.daily_climatology(e_u60, CLIM_YEARS)
clim_pcz, std_pcz = sc.daily_climatology(e_pcz, CLIM_YEARS)
clim_vt, _ = sc.daily_climatology(e_vt, CLIM_YEARS)
clim_pct, _ = sc.daily_climatology(e_pct, CLIM_YEARS)
e_pch = (e_pcz - on_days(clim_pcz, e_pcz)) / on_days(std_pcz, e_pcz)    # standardized polar-cap height

# calendar-month means and their 1981-2010 climatology (U, T, Z) for the model drift correction
e_month_clim = sc.obs_month_climatology(era5[["U", "T", "Z"]], CLIM_YEARS).load()   # (month, plev, lat)
e_u60_month = sc.monthly_means(e_u60)

### 2. Model

* **Monthly zonal means**: the Reanalysis 1981-2010 May/Nov starts (drift climatology and the
  model's own interannual spread) and each 2026 start (pooled members).
* **Daily zonal means** of each 2026 start (pooled members) from the native `eam.h1` files, with the
  monthly lead-time climatology interpolated to days. The first build reads about 90 x 25 history
  files (about 15 min with 8 processes); later runs read the caches.

In [ ]:
%%time
ref = {m: sc.monthly_zonal(rc.REFERENCE["case_prefix"], "Reanalysis", range(CLIM_YEARS[0], CLIM_YEARS[1] + 1), m,
                           mode=derivation_mode, workers=WORKERS)
       for m in rc.REFERENCE["init_months"]}
MPLEV = ref[rc.REFERENCE["init_months"][0]]["plev"].values
e_month_clim_m = e_month_clim.sel(plev=MPLEV)            # ERA5 on the model monthly levels (all are ERA5 levels)
lead_clim = {init: sc.lead_climatology(init, ref, e_month_clim_m, method=CLIM_METHOD) for init in INIT_MONTHS}


def source_of(prefix):
    return prefix.split(f"{MODEL_PREFIX}_")[-1]


def target_time(init, nlead=rc.NLEAD):
    out = []
    for lead in range(nlead):
        m = init - 1 + lead
        out.append(np.datetime64(f"{YEAR + m // 12:04d}-{m % 12 + 1:02d}-15"))
    return np.asarray(out, dtype="datetime64[ns]")


def pooled_monthly(init):
    # pooled monthly zonal means (member, lead, plev, lat) of one start; repeats counted once
    parts = []
    for name, prefix in POOL.items():
        ds = sc.monthly_zonal(prefix, source_of(prefix), [YEAR], init, mode=derivation_mode, workers=WORKERS)
        if ds is None:
            continue
        ds = ds.isel(year=0, drop=True)
        ds = ds.isel(member=np.flatnonzero(np.isfinite(ds["U"].sel(plev=REF_LEVEL).isel(lead=0, lat=-1)).values))
        u = ds["U"].sel(plev=REF_LEVEL).values
        if any(p.sizes["member"] == ds.sizes["member"] and np.array_equal(u, p["U"].sel(plev=REF_LEVEL).values, equal_nan=True)
               for p in parts):
            print(f"[monthly {name} init{init:02d}] repeats an already pooled case; not pooled again")
            continue
        parts.append(ds.assign_coords(member=[f"{name}:{m}" for m in ds["member"].values]))
    out = xr.concat(parts, dim="member")
    return out.assign_coords(time=("lead", target_time(init, out.sizes["lead"])))


monthly = {init: pooled_monthly(init) for init in INIT_MONTHS}
daily = {init: sc.pooled_daily_zonal(SIMULATION_ROOT, POOL, YEAR, init, mode=derivation_mode, workers=WORKERS)
         for init in INIT_MONTHS}
for init in INIT_MONTHS:
    d = daily[init]
    print(f"[init {MONTH_NAMES[init]} {YEAR}] {d.sizes['member']} members, daily "
          f"{pd.Timestamp(d['time'].values[0]):%d %b %Y} - {pd.Timestamp(d['time'].values[-1]):%d %b %Y}; "
          f"monthly {monthly[init].sizes['member']} members; climatology: {lead_clim[init].attrs['climatology_method']}")

In [ ]:
# Daily model indices, the model climatology on the same days, and the ERA5 climatology on those days
m_u60, m_u60_clim, m_u60_adj, m_pch, m_vt, z_offset = {}, {}, {}, {}, {}, {}
std_pcz_m = sc.interp_logp(std_pcz, sc.DAILY_LEVELS)
for init in INIT_MONTHS:
    d = daily[init]
    lc = sc.interp_logp(lead_clim[init][["U", "Z"]], sc.DAILY_LEVELS)
    # native daily Z is ~40 m (10 hPa) to ~200 m (1000 hPa) above the monthly Z3 the climatology is made
    # from (online vs post-hoc vertical interpolation, extrapolation below ground); U and T agree: add
    # the ensemble-mean daily-minus-monthly offset of the same runs
    z_offset[init] = sc.daily_monthly_offset(d, monthly[init], YEAR, init)
    lc["Z"] = lc["Z"] + z_offset[init]
    cd = sc.daily_from_lead(lc, YEAR, init, d["time"].values)
    cd = cd.assign_coords(time=d["time"])
    m_u60[init] = sc.lat_mean(d["U"].sel(plev=REF_LEVEL), sc.U60)                      # (member, time)
    m_u60_clim[init] = sc.lat_mean(cd["U"].sel(plev=REF_LEVEL), sc.U60)                # (time)
    # bias-adjusted: remove the model-minus-ERA5 climatological difference of each day
    m_u60_adj[init] = m_u60[init] - (m_u60_clim[init] - on_days(clim_u60, m_u60_clim[init]))
    pcz = sc.lat_mean(d["Z"], sc.PCAP) - sc.lat_mean(cd["Z"], sc.PCAP)                  # (member, time, plev)
    m_pch[init] = (pcz / on_days(std_pcz_m, pcz)).dropna("plev", how="all")
    m_vt[init] = sc.lat_mean(d["VT"].sel(plev=100.0), sc.VT_BAND)

# Monthly model indices: U60 at REF_LEVEL (raw and anomaly) and equatorial U (QBO)
mm_u60, mm_u60_anom, mm_qbo = {}, {}, {}
for init in INIT_MONTHS:
    mo = monthly[init]
    mm_u60[init] = sc.lat_mean(mo["U"].sel(plev=REF_LEVEL), sc.U60)
    mm_u60_anom[init] = mm_u60[init] - sc.lat_mean(lead_clim[init]["U"].sel(plev=REF_LEVEL), sc.U60)
    mm_qbo[init] = sc.lat_mean(mo["U"], sc.EQ)                                          # (member, lead, plev)

# Check of the daily-minus-monthly Z offset: a processing difference should be nearly constant across
# leads (seasons) and starts; a physical signal would vary with the season and the initial state
rows = []
for init, off in z_offset.items():
    for p in (10.0, 50.0, 100.0, 500.0, 1000.0):
        v = sc.lat_mean(off.sel(plev=p), sc.PCAP)
        rows.append({"start": MONTH_NAMES[init], "plev (hPa)": p, "mean (m)": float(v.mean()),
                     "min over leads": float(v.min()), "max over leads": float(v.max()), "std over leads": float(v.std())})
print("Polar-cap (60-90N) daily-minus-monthly Z offset by lead (ensemble mean), per start:")
display(pd.DataFrame(rows).pivot_table(index="plev (hPa)", columns="start", values=["mean (m)", "std over leads"]).round(1))


### 3. Model stratospheric climatology against ERA5

Monthly U(60°N, 10 hPa) and 10-hPa polar-cap temperature by calendar month: ERA5 1981-2010
(mean and ±1 standard deviation of the monthly means) and the Reanalysis hindcasts at each
lead (May and Nov starts, all years and members, so the shading is the model's total spread). Right:
DJF zonal-mean U, the model at leads 1-3 of the Nov starts minus ERA5 (shading) with ERA5 contours.

The table below the figure compares the DJF vortex strength and its interannual standard deviation: E3SM's vortex is weaker (about 23 vs 29 m/s) and varies much less from winter to winter (about 4 vs 10 m/s), so the model tercile bounds in section 6 are narrow, and a few m/s of anomaly moves most members out of the middle tercile.

In [ ]:
fig = plt.figure(figsize=(17, 5.2))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.15], wspace=0.28)
ref_colors = {5: "tab:orange", 11: "tab:purple"}
for k, (field, name, units) in enumerate([("U", "U(60°N, 10 hPa)", "m s$^{-1}$"), ("T", "T(60-90°N, 10 hPa)", "K")]):
    ax = fig.add_subplot(gs[k])
    band = sc.U60 if field == "U" else sc.PCAP
    e_series = sc.monthly_means(sc.lat_mean(era5[field].sel(plev=REF_LEVEL), band))
    e_series = e_series.sel(time=slice(f"{CLIM_YEARS[0]}-01", f"{CLIM_YEARS[1]}-12"))
    g = e_series.groupby("time.month")
    order = [7, 8, 9, 10, 11, 12, 1, 2, 3, 4, 5, 6]
    x = np.arange(12)
    mean, std = g.mean().sel(month=order).values, g.std().sel(month=order).values
    ax.fill_between(x, mean - std, mean + std, color="0.8", label="ERA5 ±1σ")
    ax.plot(x, mean, "k-", lw=2.5, label=f"ERA5 {CLIM_YEARS[0]}-{CLIM_YEARS[1]}")
    for m, ds in ref.items():
        idx = sc.lat_mean(ds[field].sel(plev=REF_LEVEL), band)                       # (year, member, lead)
        months = sc.target_months(m)
        for half in (0, 1):                                                          # leads 0-11, 12-23
            leads = np.arange(12 * half, 12 * half + 12)
            xs = np.array([order.index(months[l]) for l in leads])
            srt = np.argsort(xs)
            mu = idx.isel(lead=leads).mean(("year", "member")).values[srt]
            sd = idx.isel(lead=leads).std(("year", "member")).values[srt]
            ls = "-" if half == 0 else ":"
            ax.plot(xs[srt], mu, ls, color=ref_colors[m], lw=2,
                    label=f"E3SM {MONTH_NAMES[m]} starts, leads {leads[0]}-{leads[-1]}")
            if half == 0:
                ax.fill_between(xs[srt], mu - sd, mu + sd, color=ref_colors[m], alpha=0.15, lw=0)
    if field == "U":
        ax.axhline(0, color="0.4", lw=0.8)
    ax.set_xticks(x, [MONTH_NAMES[m][0] for m in order])
    ax.set_title(f"({'ab'[k]}) {name}", loc="left")
    ax.set_ylabel(units)
    ax.grid(alpha=0.3)
    if k == 0:
        ax.legend(loc="upper left", fontsize=fontz * 0.65, frameon=False)

ax = fig.add_subplot(gs[2])
nov = ref[11]["U"].isel(lead=[1, 2, 3]).mean(("year", "member", "lead"))
obs_djf = e_month_clim_m["U"].sel(month=[12, 1, 2]).mean("month")
diff = (nov - obs_djf).sel(lat=slice(0, 90))
levels = np.arange(-12, 13, 2)
cf = ax.contourf(diff["lat"], diff["plev"], diff, levels=levels, cmap="RdBu_r", extend="both")
cs = ax.contour(obs_djf["lat"].sel(lat=slice(0, 90)), obs_djf["plev"], obs_djf.sel(lat=slice(0, 90)),
                levels=np.arange(-40, 81, 10), colors="k", linewidths=0.8)
ax.clabel(cs, fmt="%d", fontsize=fontz * 0.6)
ax.set_yscale("log")
ax.set_ylim(1000, 1)
ax.set_yticks([1000, 300, 100, 30, 10, 3, 1], ["1000", "300", "100", "30", "10", "3", "1"])
ax.set_xlabel("Latitude (°N)")
ax.set_ylabel("hPa")
ax.set_title("(c) DJF zonal-mean U: E3SM (Nov starts, leads 1-3) − ERA5", loc="left", fontsize=fontz * 0.9)
fig.colorbar(cf, ax=ax, label="m s$^{-1}$", shrink=0.9)
save(fig, "climatology", "u60_t60_10hPa", "djf_u_bias")
plt.show()

clim_table = pd.DataFrame({
    "ERA5": e_u60_month.sel(time=slice(f"{CLIM_YEARS[0]}-01", f"{CLIM_YEARS[1]}-12")).groupby("time.month").mean().to_series(),
    **{f"E3SM {MONTH_NAMES[m]} starts (leads 0-11)": pd.Series(
        sc.lat_mean(ds["U"].sel(plev=REF_LEVEL), sc.U60).isel(lead=slice(0, 12)).mean(("year", "member")).values,
        index=sc.target_months(m)[:12]) for m, ds in ref.items()},
}).loc[[10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]].round(1)
clim_table.index = [MONTH_NAMES[m] for m in clim_table.index]
print(f"U(60N, {REF_LEVEL:g} hPa) monthly climatology, m/s")
display(clim_table.T)

# DJF vortex strength and its interannual variability: the model spread sets the width of its tercile bounds (section 6)
W_DJF = xr.DataArray([31.0, 31.0, 28.0], dims="lead")
var_rows = []
for m, ds in ref.items():
    leads = [(12 - m) % 12, (12 - m) % 12 + 1, (12 - m) % 12 + 2]
    u = sc.lat_mean(ds["U"].sel(plev=REF_LEVEL), sc.U60).isel(lead=leads)
    djf = (u * W_DJF.assign_coords(lead=u["lead"])).sum("lead") / 90.0
    var_rows.append({"set": f"E3SM {MONTH_NAMES[m]} starts (leads {leads[0]}-{leads[-1]})", "DJF mean (m/s)": float(djf.mean()),
                     "total sd": float(djf.std()), "sd of yearly ensemble means": float(djf.mean("member").std("year")),
                     "within-year member sd": float(djf.std("member").mean())})
e_d = [float((e_u60_month.sel(time=[np.datetime64(f"{y}-12-15"), np.datetime64(f"{y + 1}-01-15"), np.datetime64(f"{y + 1}-02-15")])
              * np.array([31.0, 31.0, 28.0])).sum() / 90.0) for y in range(CLIM_YEARS[0], CLIM_YEARS[1])]
var_rows.append({"set": f"ERA5 {CLIM_YEARS[0]}/{str(CLIM_YEARS[0] + 1)[2:]}-{CLIM_YEARS[1] - 1}/{str(CLIM_YEARS[1])[2:]}",
                 "DJF mean (m/s)": np.mean(e_d), "total sd": np.std(e_d, ddof=1)})
print(f"DJF U(60N, {REF_LEVEL:g} hPa): mean and interannual standard deviation")
display(pd.DataFrame(var_rows).round(2))


### 4. QBO

Monthly zonal-mean U over 5°S-5°N: ERA5 (top), then the ensemble mean of each start with the ERA5
values over the same months as contours. The right panels show the `QBO_LEVEL` wind of every member
against ERA5. A forecast QBO that decays, or that stops descending, moves the Holton-Tan pathway
(easterly lower-stratospheric QBO, weaker vortex) away from the observed one.

In [ ]:
qbo_span = (np.datetime64(f"{YEAR - 1}-01-01"), np.datetime64(f"{YEAR + 2}-08-01"))
fig, axes = plt.subplots(len(INIT_MONTHS) + 1, 2, figsize=(17, 3.0 * (len(INIT_MONTHS) + 1)),
                         gridspec_kw={"width_ratios": [1.6, 1], "hspace": 0.45, "wspace": 0.18}, sharex="col")
qlev = np.arange(-35, 36, 5)
eq = e_qbo.sel(time=slice(*qbo_span), plev=slice(1, 100))
ax = axes[0, 0]
cf = ax.contourf(eq["time"], eq["plev"], eq.T, levels=qlev, cmap="RdBu_r", extend="both")
ax.set_title(f"(a) ERA5 zonal-mean U, 5°S-5°N (through {obs_end:%b %Y})", loc="left")
axes[0, 1].axis("off")
for k, init in enumerate(INIT_MONTHS, start=1):
    q = mm_qbo[init].sel(plev=slice(1, 100))
    qt = q.assign_coords(time=monthly[init]["time"]).swap_dims(lead="time")
    ax = axes[k, 0]
    ax.contourf(qt["time"], qt["plev"], qt.mean("member").T, levels=qlev, cmap="RdBu_r", extend="both")
    ov = eq.sel(time=slice(qt["time"].values[0], qt["time"].values[-1]))
    if ov.sizes["time"] > 1:
        cs = ax.contour(ov["time"], ov["plev"], ov.T, levels=[-20, -10, 0, 10], colors="k", linewidths=0.8)
        ax.clabel(cs, fmt="%d", fontsize=fontz * 0.55)
    ax.axvline(np.datetime64(f"{YEAR}-{init:02d}-01"), color="k", lw=1.5)
    ax.set_title(f"({'abcdefgh'[2 * k - 1]}) E3SM {MONTH_NAMES[init]} {YEAR} start, ensemble mean "
                 f"({monthly[init].sizes['member']} members; contours: ERA5)", loc="left", fontsize=fontz * 0.85)
    ax2 = axes[k, 1]
    q50 = qt.sel(plev=QBO_LEVEL)
    for mem in q50["member"].values:
        ax2.plot(q50["time"], q50.sel(member=mem), color=INIT_COLORS[init], lw=0.6, alpha=0.5)
    ax2.plot(q50["time"], q50.mean("member"), color=INIT_COLORS[init], lw=2.5)
    e50 = e_qbo.sel(plev=QBO_LEVEL, time=slice(*qbo_span))
    ax2.plot(e50["time"], e50, "k-", lw=2)
    ax2.axhline(0, color="0.4", lw=0.8)
    ax2.set_ylim(-35, 25)
    ax2.set_title(f"({'abcdefgh'[2 * k]}) U {QBO_LEVEL:g} hPa, 5°S-5°N", loc="left", fontsize=fontz * 0.85)
    ax2.grid(alpha=0.3)
for ax in axes[:, 0]:
    ax.set_yscale("log")
    ax.set_ylim(100, 1)
    ax.set_yticks([100, 50, 30, 10, 3, 1], ["100", "50", "30", "10", "3", "1"])
    ax.set_ylabel("hPa")
    ax.set_xlim(*qbo_span)
for ax in axes[-1]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
fig.colorbar(cf, ax=axes[:, 0], label="m s$^{-1}$", shrink=0.6, pad=0.01)
save(fig, "qbo", YEAR)
plt.show()

# QBO phase before each forecast winter: member QBO_LEVEL wind over QBO_MONTHS
qbo_rows = []
for init in INIT_MONTHS:
    qt = mm_qbo[init].sel(plev=QBO_LEVEL).assign_coords(time=monthly[init]["time"]).swap_dims(lead="time")
    for w in (YEAR, YEAR + 1):
        sel = [t for t in pd.DatetimeIndex(qt["time"].values) if t.year == w and t.month in QBO_MONTHS]
        if len(sel) == len(QBO_MONTHS):
            v = qt.sel(time=sel).mean("time")
            qbo_rows.append({"start": MONTH_NAMES[init], "winter": f"{w}/{str(w + 1)[2:]}",
                             f"U{QBO_LEVEL:g} mean (m/s)": float(v.mean()), "members easterly": int((v < 0).sum()),
                             "members": int(v.size)})
e_qbo_on = {w: float(e_qbo.sel(plev=QBO_LEVEL).sel(time=[np.datetime64(f"{w}-{m:02d}-15") for m in QBO_MONTHS]).mean())
            for w in range(ERA5_FIRST_YEAR, YEAR + 1)
            if pd.Timestamp(f"{w}-{max(QBO_MONTHS):02d}-15") <= obs_end}
print(f"QBO {QBO_LEVEL:g} hPa over months {QBO_MONTHS}; ERA5 last observed autumn {max(e_qbo_on)}: {e_qbo_on[max(e_qbo_on)]:+.1f} m/s")
display(pd.DataFrame(qbo_rows).round(1))

### 5. Winter 2025/26 and the 1 Feb 2026 start

Observed daily U(60°N, 10 hPa) from 1 Oct 2025 with the ERA5 1981-2010 daily climatology
(mean ±1σ and ±2σ), the members and ensemble mean of the 1 Feb 2026 start, and the observed and
member final-warming dates (bottom rug). Below: the observed standardized polar-cap height and the Feb
start's ensemble mean over the same days.

In [ ]:
w0 = YEAR - 1
span = (pd.Timestamp(w0, 10, 1), pd.Timestamp(YEAR, 6, 30))
e_obs = e_u60.sel(time=slice(*span))
cl = on_days(clim_u60, e_obs)
sd = on_days(std_u60, e_obs)
obs_ssw_w0 = [d for d in sc.ssw_central_dates(e_u60.values, e_u60["time"].values, **SSW) if sc.winter_of(d) == w0]
obs_fw = {y: sc.final_warming_date(e_u60.values, e_u60["time"].values, y, recover_days=SSW["recover_days"])
          for y in range(ERA5_FIRST_YEAR, obs_end.year + 1)}
fw_date, fw_status = sc.final_warming(e_u60.values, e_u60["time"].values, YEAR, recover_days=SSW["recover_days"])
fw_text = f"{fw_date:%d %b} ({fw_status}: ERA5 ends {obs_end:%d %b %Y})" if pd.notna(fw_date) else f"none yet ({fw_status})"
print(f"ERA5 winter {w0}/{str(YEAR)[2:]}: SSW central dates {[f'{d:%d %b %Y}' for d in obs_ssw_w0] or 'none'}; "
      f"final warming {fw_text}")

init = 2
has_feb = init in INIT_MONTHS
fig, axes = plt.subplots(3 if has_feb else 2, 1, figsize=(15, 12 if has_feb else 8.5),
                         gridspec_kw={"height_ratios": [1.3, 1, 1][: 3 if has_feb else 2], "hspace": 0.32}, sharex=True)
ax = axes[0]
t = e_obs["time"].values
ax.fill_between(t, cl - 2 * sd, cl + 2 * sd, color="0.9", label="ERA5 ±2σ")
ax.fill_between(t, cl - sd, cl + sd, color="0.78", label="ERA5 ±1σ")
ax.plot(t, cl, color="0.4", lw=1.5, label=f"ERA5 {CLIM_YEARS[0]}-{CLIM_YEARS[1]} mean")
if has_feb:
    u = m_u60[init].sel(time=slice(*span))
    for mem in u["member"].values:
        ax.plot(u["time"], u.sel(member=mem), color=INIT_COLORS[init], lw=0.6, alpha=0.45)
    ax.plot(u["time"], u.mean("member"), color=INIT_COLORS[init], lw=2.5, label=f"E3SM 1 Feb {YEAR} ({u.sizes['member']} members)")
    ax.plot(u["time"], m_u60_clim[init].sel(time=slice(*span)), "--", color=INIT_COLORS[init], lw=1.5,
            label="E3SM lead-time climatology")
ax.plot(t, e_obs, "k-", lw=2.5, label="ERA5")
for d in obs_ssw_w0:
    ax.axvline(d, color="k", ls=":", lw=1.2)
    ax.text(d, 55, f" SSW {d:%d %b}", fontsize=fontz * 0.7, va="top")
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("m s$^{-1}$")
ax.set_ylim(-30, 60)
ax.set_title(f"(a) U(60°N, {REF_LEVEL:g} hPa), winter {w0}/{str(YEAR)[2:]}", loc="left")
# final-warming rug
if has_feb:
    fw_m = [sc.final_warming_date(m_u60[init].sel(member=mem).values, m_u60[init]["time"].values, YEAR,
                                  recover_days=SSW["recover_days"]) for mem in m_u60[init]["member"].values]
    fw_m = [d for d in fw_m if pd.notna(d)]
    ax.plot(fw_m, np.full(len(fw_m), -27), "|", color=INIT_COLORS[init], ms=14, mew=2, label="final warming (members)")
if pd.notna(obs_fw.get(YEAR, pd.NaT)):
    ax.plot([obs_fw[YEAR]], [-27], "kv", ms=11, label="final warming (ERA5)")
ax.legend(loc="upper right", ncol=2, fontsize=fontz * 0.65, frameon=False)
ax.grid(alpha=0.3)

pch_lev = np.array([-2.5, -2, -1.5, -1, -0.5, 0.5, 1, 1.5, 2, 2.5])
ep = e_pch.sel(time=slice(*span)).sel(plev=slice(1, 1000))
ax = axes[1]
cf = ax.contourf(ep["time"], ep["plev"], ep.T, levels=pch_lev, cmap="RdBu_r", extend="both")
ax.set_title("(b) ERA5 polar-cap (60-90°N) height anomaly / σ", loc="left")
if has_feb:
    mp = m_pch[init].sel(time=slice(*span)).mean("member")
    ax = axes[2]
    ax.contourf(mp["time"], mp["plev"], mp.T, levels=pch_lev, cmap="RdBu_r", extend="both")
    ax.set_title(f"(c) E3SM 1 Feb {YEAR} ensemble mean, same quantity (ERA5 σ)", loc="left")
for ax in axes[1:]:
    ax.set_yscale("log")
    ax.set_ylim(1000, 1)
    ax.set_yticks([1000, 300, 100, 30, 10, 1], ["1000", "300", "100", "30", "10", "1"])
    ax.set_ylabel("hPa")
axes[-1].set_xlim(*span)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%Y"))
fig.colorbar(cf, ax=axes[1:], label="σ", shrink=0.8, pad=0.01)
save(fig, "winter", f"{w0}{YEAR}", "u60n10_pch")
plt.show()

### 6. The 2026/27 and 2027/28 winters

One row per start: every member's daily U(60°N, 10 hPa) (thin), the ensemble mean (thick), the
model's lead-time climatology (dashed) and ERA5 (climatology band and observations). Dots mark the
SSW central dates on the raw model wind.

In [ ]:
plume_span = (pd.Timestamp(YEAR, 8, 1), pd.Timestamp(YEAR + 2, 5, 31))
fig, axes = plt.subplots(len(INIT_MONTHS), 1, figsize=(16, 3.6 * len(INIT_MONTHS)), sharex=True, sharey=True,
                         gridspec_kw={"hspace": 0.25})
days = pd.date_range(*plume_span, freq="D")
dummy = xr.DataArray(np.zeros(days.size), dims="time", coords={"time": days.values})
cl, sd = on_days(clim_u60, dummy), on_days(std_u60, dummy)
model_ssw = {}                        # init -> DataFrame (series, winter, central_date) on the raw wind
model_ssw_adj = {}                    # bias-adjusted
for ax, init in zip(np.atleast_1d(axes), INIT_MONTHS):
    u = m_u60[init]
    ax.fill_between(days, cl - sd, cl + sd, color="0.82", label="ERA5 ±1σ")
    ax.plot(days, cl, color="0.35", lw=1.5, label="ERA5 climatology")
    for mem in u["member"].values:
        ax.plot(u["time"], u.sel(member=mem), color=INIT_COLORS[init], lw=0.5, alpha=0.35)
    ax.plot(u["time"], u.mean("member"), color=INIT_COLORS[init], lw=2.5, label="ensemble mean")
    ax.plot(u["time"], m_u60_clim[init], "--", color="k", lw=1.5, label="E3SM lead-time climatology")
    e_part = e_u60.sel(time=slice(*plume_span))
    if e_part.sizes["time"]:
        ax.plot(e_part["time"], e_part, "k-", lw=2.5, label="ERA5")
    tabs, tabs_adj = [], []
    for mem in u["member"].values:
        tabs.append(sc.ssw_table(u.sel(member=mem).values, u["time"].values, str(mem), **SSW))
        tabs_adj.append(sc.ssw_table(m_u60_adj[init].sel(member=mem).values, u["time"].values, str(mem), **SSW))
    model_ssw[init] = pd.concat(tabs, ignore_index=True)
    model_ssw_adj[init] = pd.concat(tabs_adj, ignore_index=True)
    ev = model_ssw[init]
    if len(ev):
        y = [float(u.sel(member=s, time=np.datetime64(d)).values) for s, d in zip(ev["series"], ev["central_date"])]
        ax.plot(ev["central_date"], np.asarray(y), "o", ms=6, mfc="w", mec="k", label="SSW (raw wind)")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylim(-35, 70)
    ax.set_ylabel("m s$^{-1}$")
    ax.grid(alpha=0.3)
    ax.set_title(f"E3SM 1 {MONTH_NAMES[init]} {YEAR} start ({u.sizes['member']} members): U(60°N, {REF_LEVEL:g} hPa)", loc="left")
np.atleast_1d(axes)[0].legend(loc="upper left", ncol=3, fontsize=fontz * 0.65, frameon=False)
np.atleast_1d(axes)[-1].set_xlim(*plume_span)
np.atleast_1d(axes)[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
save(fig, "plume", "u60n10", YEAR)
plt.show()

**SSW occurrence in the forecasts.** For every fully covered winter (1 Nov - 30 Apr) of each start:
the probability of at least one SSW and the events per member-winter, **by start and by case**
(the `POOL` cases are different initial-state setups, not interchangeable members), then pooled
over the cases with their member counts as weights. Detection on the **raw** model wind is the primary
catalogue; the **bias-adjusted** wind is a threshold-sensitivity test only. 90% intervals come from a
bootstrap over members (one member-winter each; the two winters of a member are never pooled), and over
winters for ERA5. ERA5 1979/80 to the last complete winter (all winters, and El Niño / La Niña winters
by DJF Niño3.4 from `NINO_PRODUCT_OBS`) is context only: two forecast winters conditioned on one
initial state are not a model climatology, so they say nothing on their own about an E3SM SSW-frequency bias.

Winter coverage: the Feb start ends 31 Jan 2028, so its 2027/28 winter is not counted; the May start
(to 30 Apr 2028) and Aug start (to 31 Jul 2028) cover both winters. The event table adds, per event,
the minimum wind, the length of the easterly spell, and the recovery date (`sc.ssw_event_details`).


In [ ]:
rng = np.random.default_rng(0)


def freq_row(label, counts, kind, **extra):
    counts = np.asarray(counts, dtype=float)
    boots = rng.choice(counts, size=(5000, counts.size)) if counts.size else np.full((1, 1), np.nan)
    return {"set": label, **extra, "wind": kind, "winters": counts.size, "events": int(np.nansum(counts)),
            "events per winter": counts.mean() if counts.size else np.nan,
            "5%": np.percentile(boots.mean(1), 5), "95%": np.percentile(boots.mean(1), 95),
            "P(≥1 SSW)": (counts > 0).mean() if counts.size else np.nan,
            "P 5%": np.percentile((boots > 0).mean(1), 5), "P 95%": np.percentile((boots > 0).mean(1), 95)}


def mid_month(times):
    # the 15th of each time's month (datetime64[ns])
    return sc.to_datetime64(times).astype("datetime64[M]").astype("datetime64[ns]") + np.timedelta64(14, "D")


def case_of(member):
    return str(member).split(":")[0]


# observed
obs_events = sc.ssw_table(e_u60.values, e_u60["time"].values, "ERA5", **SSW)
obs_winters = sc.covered_winters(e_u60["time"].values)
obs_counts = pd.Series(0, index=obs_winters).add(obs_events.groupby("winter").size(), fill_value=0).astype(int)
nino_obs = rc.box_mean(rc.obs_anomaly("TS", years=(ERA5_FIRST_YEAR, YEAR), mode=derivation_mode, product=NINO_PRODUCT_OBS),
                       rc.NINO_REGIONS["Nino3.4"]).load()
nino_obs = nino_obs.assign_coords(time=mid_month(nino_obs["time"].values))


def winter_mean(series, w, months=DJF):
    times = [np.datetime64(f"{w + dy}-{m:02d}-15") for dy, m in months]
    try:
        v = series.sel(time=times)
    except KeyError:
        return np.nan
    return float(v.mean()) if np.isfinite(v).all() else np.nan


obs_djf_nino = pd.Series({w: winter_mean(nino_obs, w) for w in obs_winters})
rows = [freq_row(f"ERA5 {obs_winters[0]}/{str(obs_winters[0] + 1)[2:]}-{obs_winters[-1]}/{str(obs_winters[-1] + 1)[2:]}",
                 obs_counts.values, "ERA5"),
        freq_row("ERA5 El Niño winters", obs_counts[obs_djf_nino >= ENSO_THRESHOLD].values, "ERA5"),
        freq_row("ERA5 La Niña winters", obs_counts[obs_djf_nino <= -ENSO_THRESHOLD].values, "ERA5")]
print("ERA5 El Niño winters:", ", ".join(f"{w}/{str(w + 1)[2:]}" for w in obs_counts[obs_djf_nino >= ENSO_THRESHOLD].index))

# model: per start x case x winter, then per start (cases pooled, member-count weights), raw and bias-adjusted
model_counts = {}                      # (init, winter, kind) -> per-member counts (all cases)
for init in INIT_MONTHS:
    winters = sc.covered_winters(m_u60[init]["time"].values)
    members = [str(m) for m in m_u60[init]["member"].values]
    for kind, tab in (("raw", model_ssw[init]), ("bias-adjusted", model_ssw_adj[init])):
        for w in winters:
            c = tab[tab["winter"] == w].groupby("series").size().reindex(members, fill_value=0)
            model_counts[(init, w, kind)] = c
            label = f"E3SM {MONTH_NAMES[init]} start, {w}/{str(w + 1)[2:]}"
            for case in dict.fromkeys(case_of(m) for m in members):
                cc = c[[case_of(m) == case for m in c.index]]
                rows.append(freq_row(label, cc.values, kind, case=case))
            weights = ", ".join(f"{k} {v}" for k, v in pd.Series([case_of(m) for m in members]).value_counts().items())
            rows.append(freq_row(label, c.values, kind, case=f"pooled ({weights})"))
for kind in ("raw", "bias-adjusted"):
    for w in (YEAR, YEAR + 1):
        pooled = [c.values for (i, ww, k), c in model_counts.items() if ww == w and k == kind]
        if pooled:
            rows.append(freq_row(f"E3SM all starts, {w}/{str(w + 1)[2:]}", np.concatenate(pooled), kind, case="pooled"))
freq = pd.DataFrame(rows)
freq.to_csv(DATA_DIR / f"ssw_frequency_{YEAR}.csv", index=False)
print(f"[WRITE] {DATA_DIR / f'ssw_frequency_{YEAR}.csv'}")
display(freq[freq["wind"] != "bias-adjusted"].round(2))
print("Threshold sensitivity (bias-adjusted wind), pooled per start:")
display(freq[(freq["wind"] == "bias-adjusted") & freq["case"].astype(str).str.startswith("pooled")].round(2))

# event catalogue with per-event properties (raw wind primary)
cat = [sc.ssw_event_details(e_u60.values, e_u60["time"].values, obs_events["central_date"], SSW["recover_days"])
       .assign(source="ERA5", series="ERA5", wind="ERA5")]
for i in INIT_MONTHS:
    for kind, tab, src in (("raw", model_ssw[i], m_u60[i]), ("bias-adjusted", model_ssw_adj[i], m_u60_adj[i])):
        for mem, ev in tab.groupby("series"):
            cat.append(sc.ssw_event_details(src.sel(member=mem).values, src["time"].values, ev["central_date"],
                                            SSW["recover_days"]).assign(source=f"E3SM init{i:02d}", series=mem, wind=kind))
events_all = pd.concat(cat, ignore_index=True)
events_all["winter"] = [sc.winter_of(d) for d in events_all["central_date"]]
events_all["case"] = [case_of(s) for s in events_all["series"]]
events_all.to_csv(DATA_DIR / f"ssw_central_dates_{YEAR}.csv", index=False)
print(f"[WRITE] {DATA_DIR / f'ssw_central_dates_{YEAR}.csv'}")
display(events_all[events_all["wind"] != "bias-adjusted"].groupby(["source", "winter"])
        [["min_u", "easterly_days", "days_to_recovery"]].median().round(1))


In [ ]:
# SSW frequency, central-date months and final-warming dates
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), gridspec_kw={"width_ratios": [1.3, 1, 1.1], "wspace": 0.3})
ax = axes[0]
sel = freq[freq["set"].str.contains("ERA5") | freq["case"].astype(str).str.startswith("pooled")]
labels = [f"{s}\n({k})" if k != "ERA5" else s for s, k in zip(sel["set"], sel["wind"])]
colors = ["k" if k == "ERA5" else ("tab:red" if k == "raw" else "tab:blue") for k in sel["wind"]]
y = np.arange(len(sel))[::-1]
for yy, (_, r), c in zip(y, sel.iterrows(), colors):   # one call per point: errorbar takes one colour
    ax.errorbar(r["events per winter"], yy, xerr=[[r["events per winter"] - r["5%"]], [r["95%"] - r["events per winter"]]],
                fmt="o", color=c, ms=8, capsize=4, zorder=3)
ax.set_yticks(y, labels, fontsize=fontz * 0.6)
ax.axvline(float(freq.iloc[0]["events per winter"]), color="k", ls=":", lw=1)
ax.set_xlabel("SSWs per winter (90% bootstrap)")
ax.set_title("(a) SSW frequency", loc="left")
ax.grid(alpha=0.3, axis="x")

ax = axes[1]
order = [11, 12, 1, 2, 3]
x = np.arange(len(order))
obs_m = obs_events["central_date"].dt.month.value_counts().reindex(order, fill_value=0) / len(obs_winters)
mod = pd.concat([model_ssw[i] for i in INIT_MONTHS])
n_mw = sum(len(c) for (i, w, k), c in model_counts.items() if k == "raw")
mod_m = (mod["central_date"].dt.month.value_counts().reindex(order, fill_value=0) / max(n_mw, 1)) if len(mod) else 0 * obs_m
ax.bar(x - 0.2, obs_m.values, 0.4, color="k", label=f"ERA5 ({len(obs_winters)} winters)")
ax.bar(x + 0.2, np.asarray(mod_m), 0.4, color="tab:red", label=f"E3SM raw ({n_mw} member-winters)")
ax.set_xticks(x, [MONTH_NAMES[m] for m in order])
ax.set_ylabel("SSWs per winter")
ax.set_title("(b) Month of the central date", loc="left")
ax.legend(frameon=False)
ax.grid(alpha=0.3, axis="y")

ax = axes[2]


def doy_spring(d):
    return (pd.Timestamp(d) - pd.Timestamp(pd.Timestamp(d).year, 1, 1)).days + 1


# only complete records enter the boxes; censored and unresolved springs (record ends before 31 Jul,
# e.g. the May start's 2028) are tabulated but not plotted
obs_fw_status = {y: sc.final_warming(e_u60.values, e_u60["time"].values, y, recover_days=SSW["recover_days"])
                 for y in range(ERA5_FIRST_YEAR, obs_end.year + 1)}
obs_fw_doy = [doy_spring(d) for d, st in obs_fw_status.values() if st == "complete"]
data, labels, cols = [obs_fw_doy], [f"ERA5\n{ERA5_FIRST_YEAR}-{obs_end.year - 1}"], ["0.6"]
fw_rows = [{"source": "ERA5", "member": "ERA5", "spring": y, "final_warming": d, "status": st}
           for y, (d, st) in obs_fw_status.items()]
for init in INIT_MONTHS:
    u = m_u60[init]
    for spring in (YEAR, YEAR + 1, YEAR + 2):
        res = [sc.final_warming(u.sel(member=m).values, u["time"].values, spring, recover_days=SSW["recover_days"])
               for m in u["member"].values]
        if all(st == "no data" for _, st in res) or (spring == YEAR and init != 2):
            continue
        fw_rows += [{"source": f"E3SM init{init:02d}", "member": str(m), "spring": spring, "final_warming": d, "status": st}
                    for m, (d, st) in zip(u["member"].values, res)]
        ok = [d for d, st in res if st == "complete"]
        n_cens = sum(st in ("censored", "unresolved") for _, st in res)
        if n_cens:
            print(f"[final warming] {MONTH_NAMES[init]} start, spring {spring}: {n_cens} of {len(res)} members censored or "
                  f"unresolved (record ends {pd.Timestamp(u['time'].values[-1]):%d %b %Y}); not plotted")
        if len(ok) >= 3:
            data.append([doy_spring(d) for d in ok])
            labels.append(f"{MONTH_NAMES[init]} start\n{spring} (n={len(ok)})")
            cols.append(INIT_COLORS[init])
bp = ax.boxplot(data, vert=False, patch_artist=True, widths=0.6, whis=(5, 95))
for patch, c in zip(bp["boxes"], cols):
    patch.set_facecolor(c)
    patch.set_alpha(0.5)
d_obs, st_obs = obs_fw_status.get(YEAR, (pd.NaT, "no data"))
if pd.notna(d_obs):
    note = "" if st_obs == "complete" else f" ({st_obs}: ERA5 ends {obs_end:%d %b})"
    ax.axvline(doy_spring(d_obs), color="k", ls="--", lw=1.5, label=f"ERA5 {YEAR}: {d_obs:%d %b}{note}")
    ax.legend(frameon=False, loc="lower right", fontsize=fontz * 0.7)
ax.set_yticks(np.arange(1, len(labels) + 1), labels, fontsize=fontz * 0.7)
ticks = [pd.Timestamp(2001, m, 1) for m in (2, 3, 4, 5, 6)]
ax.set_xticks([doy_spring(t) for t in ticks], [f"1 {t:%b}" for t in ticks])
ax.set_title("(c) Final-warming date", loc="left")
ax.grid(alpha=0.3, axis="x")
save(fig, "ssw", "frequency", "final_warming", YEAR)
plt.show()

fw_table = pd.DataFrame(fw_rows)
fw_table.to_csv(DATA_DIR / f"final_warming_dates_{YEAR}.csv", index=False)
print(f"[WRITE] {DATA_DIR / f'final_warming_dates_{YEAR}.csv'}")

**DJF vortex strength.** Drift-corrected, day-weighted DJF-mean U(60°N, 10 hPa) anomalies of every
member (monthly output, the product the drift climatology is made from), with **uncalibrated ensemble
tercile probabilities**: there is no hindcast verification of these probabilities.

* **Tercile bounds** are matched to the start and lead: the May start uses the Reanalysis May starts
  at the same leads (all years and members). The Feb and Aug starts have no Reanalysis, so their bounds
  are a **surrogate**: the May and Nov Reanalysis starts pooled at the leads of the same target winter.
* **Drift-correction sensitivity** (Feb/Aug only; the May climatology is direct): the default
  `lead_drift`, the `target_month` alternative, and `lead_drift` with the May-only or Nov-only drift.
  What matters is whether the sign and ranking of the anomalies survive these choices.
* ERA5 DJF anomalies (1979/80 on, against the 1981-2010 calendar-month climatology) are shown for
  reference, with El Niño winters marked.


In [ ]:
DJF_DAYS = np.array([31.0, 31.0, 28.0])          # day weights of Dec, Jan, Feb (no-leap model calendar)


def djf_mean(series, w):
    # day-weighted DJF mean of a monthly series stamped on the 15th; NaN unless all three months exist
    times = [np.datetime64(f"{w + dy}-{m:02d}-15") for dy, m in DJF]
    if not all(t in series["time"].values for t in times):
        return None
    v = series.sel(time=times)
    return (v * xr.DataArray(DJF_DAYS, dims="time", coords={"time": v["time"]})).sum("time") / DJF_DAYS.sum()


def djf_leads(start_month, k):
    # leads of Dec, Jan, Feb of the k-th winter after a start month
    dec = (12 - start_month) % 12 + 12 * k
    return [dec, dec + 1, dec + 2]


def ref_djf_anomalies(m, k):
    # Reanalysis DJF anomalies (year, member) of start month m in its k-th winter, about the mean over years and members
    leads = djf_leads(m, k)
    if leads[-1] >= ref[m].sizes["lead"]:
        return None
    u = sc.lat_mean(ref[m]["U"].sel(plev=REF_LEVEL), sc.U60).isel(lead=leads)
    u = (u * xr.DataArray(DJF_DAYS, dims="lead", coords={"lead": u["lead"]})).sum("lead") / DJF_DAYS.sum()
    return (u - u.mean()).values.ravel()


def tercile_bounds(init, k):
    # (lo, hi, kind): matched to the start and lead when the Reanalysis has the start month, else a surrogate
    if init in ref:
        a, kind = ref_djf_anomalies(init, k), "matched"
    else:
        parts = [ref_djf_anomalies(m, k) for m in ref]
        a, kind = np.concatenate([p for p in parts if p is not None]), "surrogate (May+Nov pooled)"
    a = a[np.isfinite(a)]
    return *np.percentile(a, [100 / 3, 200 / 3]), kind, a.size


# observed DJF anomalies: calendar-month climatology subtracted month by month
era5_month_clim = e_u60_month.sel(time=slice(f"{CLIM_YEARS[0]}-01-01", f"{CLIM_YEARS[1]}-12-31")).groupby("time.month").mean("time")
e_anom_month = (e_u60_month.groupby("time.month") - era5_month_clim).drop_vars("month")
assert e_anom_month.dims == ("time",), e_anom_month.dims
e_djf = pd.Series({w: float(v) for w in range(ERA5_FIRST_YEAR, YEAR) if (v := djf_mean(e_anom_month, w)) is not None}).dropna()
e_lo, e_hi = np.percentile(e_djf.values, [100 / 3, 200 / 3])

# drift-correction alternatives for the DJF climatology (Feb/Aug); the May climatology is direct
CLIM_VARIANTS = {"lead_drift": (CLIM_METHOD, list(ref)), "target_month": ("target_month", list(ref)),
                 "lead_drift, May drift only": ("lead_drift", [5]), "lead_drift, Nov drift only": ("lead_drift", [11])}

m_djf, prob_rows = {}, []
for init in INIT_MONTHS:
    raw = mm_u60[init].assign_coords(time=monthly[init]["time"]).swap_dims(lead="time")
    for k, w in enumerate((YEAR, YEAR + 1)):
        v_raw = djf_mean(raw, w)
        if v_raw is None:
            continue
        lo, hi, kind, n_ref = tercile_bounds(init, k)
        for name, (method, refs) in CLIM_VARIANTS.items():
            if init in ref and name != "lead_drift":
                continue                                           # direct climatology: no alternative
            lc = sc.lead_climatology(init, {m: ref[m] for m in refs}, e_month_clim_m, method=method)
            clim = sc.lat_mean(lc["U"].sel(plev=REF_LEVEL), sc.U60).assign_coords(time=("lead", monthly[init]["time"].values))
            v = v_raw - djf_mean(clim.swap_dims(lead="time"), w)
            if name == "lead_drift":
                m_djf[(init, w)] = v
            prob_rows.append({"start": MONTH_NAMES[init], "winter": f"{w}/{str(w + 1)[2:]}", "climatology": name,
                              "members": int(v.size), "mean anomaly (m/s)": float(v.mean()),
                              "P(weak)": float((v < lo).mean()), "P(normal)": float(((v >= lo) & (v <= hi)).mean()),
                              "P(strong)": float((v > hi).mean()), "bounds": f"{lo:+.1f} / {hi:+.1f} ({kind}, n={n_ref})"})
probs = pd.DataFrame(prob_rows)
probs.to_csv(DATA_DIR / f"djf_vortex_terciles_{YEAR}.csv", index=False)
print(f"[WRITE] {DATA_DIR / f'djf_vortex_terciles_{YEAR}.csv'}; ERA5 tercile bounds {e_lo:+.1f} / {e_hi:+.1f} m/s")
nino_years = [w for w in e_djf.index if obs_djf_nino.get(w, np.nan) >= ENSO_THRESHOLD]
print(f"ERA5 El Niño winters: weak {np.mean(e_djf[nino_years] < e_lo):.0%}, strong {np.mean(e_djf[nino_years] > e_hi):.0%} "
      f"({len(nino_years)} winters)")
print("Uncalibrated ensemble tercile probabilities (default climatology first; Feb/Aug rows below it test the drift correction):")
display(probs.round(2))

fig, ax = plt.subplots(figsize=(13, 5))
pos, labels = [], []
for k, ((init, w), v) in enumerate(m_djf.items()):
    xs = np.full(v.size, k) + rng.uniform(-0.15, 0.15, v.size)
    ax.scatter(xs, v.values, color=INIT_COLORS[init], s=18, alpha=0.7)
    ax.plot([k - 0.3, k + 0.3], [float(v.mean())] * 2, color=INIT_COLORS[init], lw=3)
    lo, hi, kind, _ = tercile_bounds(init, w - YEAR)
    ax.plot([k - 0.35, k + 0.35], [lo, lo], color="0.4", lw=1, ls="--" if kind != "matched" else "-")
    ax.plot([k - 0.35, k + 0.35], [hi, hi], color="0.4", lw=1, ls="--" if kind != "matched" else "-")
    pos.append(k)
    labels.append(f"{MONTH_NAMES[init]} start\n{w}/{str(w + 1)[2:]}")
k0 = len(pos) + 0.5
xs = np.full(e_djf.size, k0) + rng.uniform(-0.15, 0.15, e_djf.size)
nino_mask = np.array([w in nino_years for w in e_djf.index])
ax.scatter(xs[~nino_mask], e_djf.values[~nino_mask], color="0.5", s=18, label="ERA5 winters")
ax.scatter(xs[nino_mask], e_djf.values[nino_mask], color="tab:orange", edgecolor="k", s=40, label="ERA5 El Niño winters")
ax.plot([k0 - 0.35, k0 + 0.35], [e_lo, e_lo], color="0.4", lw=1)
ax.plot([k0 - 0.35, k0 + 0.35], [e_hi, e_hi], color="0.4", lw=1)
pos.append(k0)
labels.append(f"ERA5\n{int(e_djf.index[0])}-{int(e_djf.index[-1]) + 1}")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(pos, labels, fontsize=fontz * 0.75)
ax.set_ylabel("DJF U(60°N, 10 hPa) anomaly (m s$^{-1}$)")
ax.set_title("DJF polar-vortex strength (drift-corrected; grey: tercile bounds, dashed = surrogate)", loc="left")
ax.legend(frameon=False, loc="lower left", fontsize=fontz * 0.7)
ax.grid(alpha=0.3, axis="y")
save(fig, "djf", "vortex", "terciles", YEAR)
plt.show()


### 7. ENSO and QBO background state of the forecasts

What tropical background the forecasts put the 2026/27 vortex in, **by start and case** (the
cases are different initial-state setups). This is a description of the forecast sample, not a test
of the ENSO and QBO teleconnections: the sampled ENSO and QBO ranges are narrow, and testing the
pathways (an average El Niño tendency to a weaker NH winter vortex; the Holton-Tan tendency to a
weaker vortex under an easterly lower-stratospheric QBO; both season- and background-dependent,
Holton and Tan 1980, Garfinkel and Hartmann 2007) needs a long hindcast record. No ENSO x QBO
interaction is fitted.

* **(a, b)** Member distributions of day-weighted DJF Niño3.4 (drift-corrected `TS`) and of the
  Oct-Nov equatorial (5°S-5°N) U at 30 and 50 hPa per start and case; ERA5 (Oct-Nov 2025, the last
  observed autumn) for reference on the QBO panel. The Niño3.4 evolution is in notebooks 1 and 8, the
  equatorial time-height sections in section 4.
* **(c, d)** Within-group centred member covariation: U'ᵢ − mean(U')_group against Xᵢ − mean(X)_group,
  so differences between starts and cases do not enter the fit.


In [ ]:
w = YEAR
groups = {}                                        # (init, case) -> DataFrame of member values
for init in INIT_MONTHS:
    if (init, w) not in m_djf:
        continue
    vortex = m_djf[(init, w)].to_series()
    vortex.index = [str(i) for i in vortex.index]
    nino = {}
    for name, prefix in POOL.items():
        anom = rc.hindcast_anomaly(prefix, init, "TS", year=YEAR, method=CLIM_METHOD, mode=derivation_mode)
        if anom is None:
            continue
        n34 = rc.box_mean(anom, rc.NINO_REGIONS["Nino3.4"]).load()
        n34 = n34.assign_coords(time=("lead", mid_month(n34["time"].values))).swap_dims(lead="time")
        for mem in n34["member"].values:
            v = djf_mean(n34.sel(member=mem), w)
            nino[f"{name}:{mem}"] = np.nan if v is None else float(v)
    q = mm_qbo[init].assign_coords(time=monthly[init]["time"]).swap_dims(lead="time")
    qt = [np.datetime64(f"{w}-{m:02d}-15") for m in QBO_MONTHS]
    qon = q.sel(time=qt).mean("time") if all(t in q["time"].values for t in qt) else None
    df = pd.DataFrame({"vortex": vortex, "nino34": pd.Series(nino).reindex(vortex.index)})
    for p in (30.0, 50.0):
        s = qon.sel(plev=p).to_series() if qon is not None else pd.Series(np.nan, index=vortex.index)
        s.index = [str(i) for i in s.index]
        df[f"qbo{p:g}"] = s.reindex(vortex.index)
    df["case"] = [case_of(m) for m in df.index]
    for case, sub in df.groupby("case"):
        groups[(init, case)] = sub

summary = pd.DataFrame([{"start": MONTH_NAMES[i], "case": c, "members": len(g),
                         "DJF Niño3.4 (°C)": g["nino34"].mean(), "ON U30 (m/s)": g["qbo30"].mean(),
                         "ON U50 (m/s)": g["qbo50"].mean(), "members U50 < 0": int((g["qbo50"] < 0).sum()),
                         "DJF vortex anomaly (m/s)": g["vortex"].mean()} for (i, c), g in groups.items()])
display(summary.round(2))
e_qbo_last = {p: float(e_qbo.sel(plev=p).sel(time=[np.datetime64(f"{max(e_qbo_on)}-{m:02d}-15") for m in QBO_MONTHS]).mean())
              for p in (30.0, 50.0)}

fig, axes = plt.subplots(2, 2, figsize=(16, 10), gridspec_kw={"hspace": 0.45, "wspace": 0.2})
keys = list(groups)
glabel = [f"{MONTH_NAMES[i]}\n{c.replace('Reanalysis-', '')}" for i, c in keys]
gcol = [INIT_COLORS[i] for i, _ in keys]
x = np.arange(len(keys))
ax = axes[0, 0]
for xi, k, c in zip(x, keys, gcol):
    v = groups[k]["nino34"].dropna().values
    ax.scatter(np.full(v.size, xi) + rng.uniform(-0.12, 0.12, v.size), v, color=c, s=16, alpha=0.7)
    ax.plot([xi - 0.3, xi + 0.3], [v.mean()] * 2, color=c, lw=3)
ax.axhline(ENSO_THRESHOLD, color="0.5", ls=":", lw=1)
ax.set_xticks(x, glabel, fontsize=fontz * 0.7)
ax.set_ylabel("DJF Niño3.4 anomaly (°C)")
ax.set_title(f"(a) ENSO state, DJF {w}/{str(w + 1)[2:]}", loc="left")
ax.grid(alpha=0.3, axis="y")
ax = axes[0, 1]
for p, dx, mk in ((30.0, -0.15, "s"), (50.0, 0.15, "o")):
    for xi, k, c in zip(x, keys, gcol):
        v = groups[k][f"qbo{p:g}"].dropna().values
        ax.scatter(np.full(v.size, xi + dx) + rng.uniform(-0.05, 0.05, v.size), v, color=c, marker=mk, s=16, alpha=0.7)
        ax.plot([xi + dx - 0.12, xi + dx + 0.12], [v.mean()] * 2, color=c, lw=3)
    ax.axhline(e_qbo_last[p], color="k", ls="--" if p == 30 else "-", lw=1.2,
               label=f"ERA5 Oct-Nov {max(e_qbo_on)} U{p:g}: {e_qbo_last[p]:+.0f} m/s")
ax.axhline(0, color="0.4", lw=0.8)
ax.set_xticks(x, glabel, fontsize=fontz * 0.7)
ax.set_ylabel("Oct-Nov U, 5°S-5°N (m s$^{-1}$)")
ax.set_title(f"(b) QBO state, Oct-Nov {w} (squares 30 hPa, circles 50 hPa)", loc="left")
ax.legend(frameon=False, fontsize=fontz * 0.65)
ax.grid(alpha=0.3, axis="y")

reg_rows = []
for ax, (xkey, xl) in zip(axes[1], [("nino34", "Niño3.4 − group mean (°C)"), ("qbo50", f"U{QBO_LEVEL:g} − group mean (m s$^{{-1}}$)")]):
    allx, ally = [], []
    for k, c in zip(keys, gcol):
        g = groups[k].dropna(subset=["vortex", xkey])
        if len(g) < 3:
            continue
        dx, dy = g[xkey] - g[xkey].mean(), g["vortex"] - g["vortex"].mean()
        ax.scatter(dx, dy, color=c, s=30, edgecolor="k", lw=0.3)
        allx += list(dx)
        ally += list(dy)
    if len(allx) > 3:
        allx, ally = np.asarray(allx), np.asarray(ally)
        r = np.corrcoef(allx, ally)[0, 1]
        b = np.polyfit(allx, ally, 1)
        xx = np.linspace(allx.min(), allx.max(), 10)
        ax.plot(xx, np.polyval(b, xx), "k-", lw=2, label=f"within-group r={r:+.2f}, slope {b[0]:+.2f} (n={allx.size})")
        reg_rows.append({"predictor": xkey, "set": f"E3SM {w}/{str(w + 1)[2:]}, within start x case", "n": allx.size,
                         "r": r, "slope": b[0]})
    ax.axhline(0, color="k", lw=0.6)
    ax.axvline(0, color="k", lw=0.6)
    ax.set_xlabel(xl)
    ax.set_ylabel("DJF U(60°N, 10 hPa) anomaly − group mean (m s$^{-1}$)")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=fontz * 0.7, loc="lower left")
axes[1, 0].set_title("(c) Member covariation with ENSO", loc="left")
axes[1, 1].set_title(f"(d) Member covariation with the QBO ({QBO_LEVEL:g} hPa)", loc="left")
handles = [Line2D([], [], color=INIT_COLORS[i], marker="o", ls="", label=f"{MONTH_NAMES[i]} start") for i in dict.fromkeys(i for i, _ in keys)]
fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False, bbox_to_anchor=(0.5, 0.0))
save(fig, "background", "enso", "qbo", YEAR)
plt.show()
display(pd.DataFrame(reg_rows).round(2))


### 8. SSW event dynamics

Lag composites around the central dates of the model SSWs (raw wind; every start, case and member-winter)
against ERA5 1979-2025.

* **Season matching.** The model events fall mostly in Feb-Mar and ERA5's in Jan-Feb, so the principal
  ERA5 composite weights each observed event by the ratio of the model and ERA5 onset-month frequencies
  (thin line: unweighted ERA5).
* **Coverage.** Only events with the whole `COMPOSITE_WINDOW` in the record enter (the count left out is printed).
* **Uncertainty.** Shading is the 90% interval of the composite **mean** from a cluster bootstrap
  (model: member trajectories, so a member's two winters stay together; ERA5: winters); dotted lines
  are the 10-90% spread of the events.
* **Heat flux** [v*T*] at 100 hPa, 45-75°N, is shown as absolute values: there is no daily model
  reference climatology for an anomaly. The model value is the eddy covariance of the daily-mean V and
  T within each latitude band; sub-daily eddies are missing (the 6-hourly output has no 100-hPa V, T).
* **Polar-cap height** (60-90°N, anomaly / ERA5 daily σ) as time-height sections, then the
  **event-level distributions** and a table of the anomaly at 10-500 hPa and of the polar-cap
  sea-level pressure over pre-onset and successive post-onset windows. The polar-cap SLP is the surface
  index; a positive post-onset anomaly is a **post-onset tropospheric association** (negative NAM),
  not by itself evidence of a causal downward influence (cf. Baldwin and Dunkerton 2001).

The model polar-cap heights use the daily-minus-monthly offset correction (section 2). Its check (the
printout there) shows it is nearly constant across leads, seasons and starts, i.e. a processing
difference and not part of the evolving signal.


In [ ]:
COMPOSITE_WINDOW = (-30, 60)          # days that must all be in the record
PRE_WINDOW, PCH_WINDOWS = (-20, -1), [(-20, -1), (0, 14), (15, 29), (30, 59)]
PCH_LEVELS = [10.0, 50.0, 100.0, 300.0, 500.0]
NBOOT = 2000

# polar-cap SLP: ERA5 and model (drift-corrected PSL climatology), standardized by the ERA5 daily sigma
e_slp = sc.era5_polar_cap_slp(ERA5_DAILY_DIR, years=(ERA5_FIRST_YEAR, YEAR))
slp_clim, slp_std = sc.daily_climatology(e_slp, CLIM_YEARS)
e_slp_s = (e_slp - on_days(slp_clim, e_slp)) / on_days(slp_std, e_slp)
m_slp_s = {}
for init in INIT_MONTHS:
    d = daily[init]
    pc = sc.lat_mean(rc.matched_climatology("PSL", init, method=CLIM_METHOD, mode=derivation_mode).mean("lon"), sc.PCAP)
    clim = sc.daily_from_lead(pc, YEAR, init, d["time"].values).assign_coords(time=d["time"])
    raw = sc.lat_mean(d["PSL"], sc.PCAP)
    m_slp_s[init] = (raw - clim) / on_days(slp_std, raw)

lags = np.arange(COMPOSITE_WINDOW[0], COMPOSITE_WINDOW[1] + 1)


def windows(series, dates):
    return sc.lag_composite(series, dates, lags)


# ERA5 composites, clusters (winters) and season-matching weights
e_dates = list(obs_events["central_date"])
e_comp = {"u": windows(e_u60, e_dates), "vt": windows(e_vt, e_dates), "pch": windows(e_pch, e_dates), "slp": windows(e_slp_s, e_dates)}
e_full = np.isfinite(e_comp["u"].values).all(axis=1)
e_months = np.array([d.month for d in e_dates])
e_clusters = np.array([sc.winter_of(d) for d in e_dates])

# model composites (raw-wind events), clusters (start x member trajectory)
m_parts, m_months, m_clusters = {k: [] for k in e_comp}, [], []
for init in INIT_MONTHS:
    for _, ev in model_ssw[init].iterrows():
        mem, d = ev["series"], ev["central_date"]
        for key, src in (("u", m_u60[init]), ("vt", m_vt[init]), ("pch", m_pch[init]), ("slp", m_slp_s[init])):
            m_parts[key].append(windows(src.sel(member=mem), [d]))
        m_months.append(d.month)
        m_clusters.append(f"{init}:{mem}")
m_comp = {k: xr.concat(v, dim="event") for k, v in m_parts.items()} if m_months else None
m_months, m_clusters = np.array(m_months), np.array(m_clusters)
m_full = np.isfinite(m_comp["u"].values).all(axis=1) if m_comp else np.array([], bool)
print(f"events with the whole {COMPOSITE_WINDOW} window: ERA5 {e_full.sum()} of {e_full.size}, "
      f"E3SM {m_full.sum()} of {m_full.size}")

# season-matching weights of the ERA5 events (model onset-month frequency / ERA5 onset-month frequency)
p_mod = pd.Series(m_months[m_full]).value_counts(normalize=True)
p_obs = pd.Series(e_months[e_full]).value_counts(normalize=True)
e_w = np.array([p_mod.get(m, 0.0) / p_obs[m] if f else 0.0 for m, f in zip(e_months, e_full)])
missing = sorted(set(p_mod.index) - set(p_obs.index))
if missing:
    print(f"[note] model onset months with no ERA5 event: {missing}; the matched ERA5 composite cannot represent them")
print("onset months, model / ERA5 (fractions):")
display(pd.DataFrame({"E3SM": p_mod, "ERA5": p_obs}).fillna(0).sort_index().round(2).T)


def comp_stats(comp, full, weights, clusters, boot=True):
    # weighted mean, cluster-bootstrap 90% CI of the mean, and the 10-90% event spread
    x = comp.values[full]
    w = np.asarray(weights, float)[full]
    mean = np.nansum(x * w.reshape((-1,) + (1,) * (x.ndim - 1)), 0) / w.sum()
    if not boot:
        return mean
    cl = np.asarray(clusters)[full]
    labs = np.unique(cl)
    idx = {l: np.flatnonzero(cl == l) for l in labs}
    boot = []
    for _ in range(NBOOT):
        ev = np.concatenate([idx[l] for l in rng.choice(labs, labs.size)])
        boot.append(np.nansum(x[ev] * w[ev].reshape((-1,) + (1,) * (x.ndim - 1)), 0) / w[ev].sum())
    boot = np.asarray(boot)
    return mean, np.nanpercentile(boot, 5, 0), np.nanpercentile(boot, 95, 0), np.nanpercentile(x, 10, 0), np.nanpercentile(x, 90, 0)


ones_m = np.ones(m_full.size)
ones_e = np.where(e_full, 1.0, 0.0)
fig = plt.figure(figsize=(17, 10.5))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.18)
for k, (key, title, units) in enumerate([("u", f"(a) U(60°N, {REF_LEVEL:g} hPa)", "m s$^{-1}$"),
                                         ("vt", "(b) [v*T*] 100 hPa, 45-75°N (absolute)", "K m s$^{-1}$")]):
    ax = fig.add_subplot(gs[0, k])
    for comp, full, w, cl, color, label in ((e_comp[key], e_full, e_w, e_clusters, "k", "ERA5, season-matched"),
                                            (m_comp[key], m_full, ones_m, m_clusters, "tab:red", "E3SM")):
        mu, lo, hi, p10, p90 = comp_stats(comp, full, w, cl)
        ax.fill_between(lags, lo, hi, color=color, alpha=0.2, lw=0)
        ax.plot(lags, mu, color=color, lw=2.5, label=f"{label} ({int(full.sum())} events)")
        ax.plot(lags, p10, color=color, lw=0.8, ls=":")
        ax.plot(lags, p90, color=color, lw=0.8, ls=":")
    mu_raw = comp_stats(e_comp[key], e_full, ones_e, e_clusters)[0]
    ax.plot(lags, mu_raw, color="k", lw=1, alpha=0.6, label="ERA5, all events (unweighted)")
    ax.axvline(0, color="0.4", lw=1)
    ax.axhline(0, color="0.4", lw=0.8)
    ax.set_xlabel("days from central date")
    ax.set_ylabel(units)
    ax.set_title(title, loc="left")
    ax.legend(frameon=False, fontsize=fontz * 0.7)
    ax.grid(alpha=0.3)
pch_lev = np.array([-2.5, -2, -1.5, -1, -0.5, 0.5, 1, 1.5, 2, 2.5])
for k, (comp, full, w, label) in enumerate(((e_comp["pch"], e_full, e_w, "ERA5, season-matched"), (m_comp["pch"], m_full, ones_m, "E3SM"))):
    ax = fig.add_subplot(gs[1, k])
    mu = xr.DataArray(comp_stats(comp, full, w, None, boot=False), dims=("lag", "plev"),
                      coords={"lag": lags, "plev": comp["plev"].values})
    c = mu.sel(plev=slice(1, 1000)).dropna("plev", how="all")
    cf = ax.contourf(c["lag"], c["plev"], c.T, levels=pch_lev, cmap="RdBu_r", extend="both")
    ax.set_yscale("log")
    ax.set_ylim(1000, 1 if label.startswith("ERA5") else 10)
    ax.set_yticks([1000, 300, 100, 30, 10, 3, 1], ["1000", "300", "100", "30", "10", "3", "1"])
    ax.axvline(0, color="k", lw=1)
    ax.set_xlabel("days from central date")
    ax.set_ylabel("hPa")
    ax.set_title(f"({'cd'[k]}) {label}: polar-cap height anomaly / σ ({int(full.sum())} events)", loc="left", fontsize=fontz * 0.9)
fig.colorbar(cf, ax=fig.axes[2:], label="σ (ERA5 daily)", shrink=0.8, pad=0.01)
save(fig, "ssw", "composite", YEAR)
plt.show()


In [ ]:
# Event-level distributions and the vertical evolution table
def event_values(comp, full, lag_range, plev=None):
    c = comp.sel(lag=slice(*lag_range))
    if plev is not None:
        c = c.sel(plev=plev)
    return c.mean("lag").values[full]


cat_raw = events_all[events_all["wind"].isin(["ERA5", "raw"])]
cat_raw = cat_raw.assign(group=np.where(cat_raw["source"] == "ERA5", "ERA5", "E3SM"))
metrics = {
    f"pre-onset [v*T*] 100 hPa, days {PRE_WINDOW[0]} to {PRE_WINDOW[1]} (K m/s)":
        (event_values(e_comp["vt"], e_full, PRE_WINDOW), event_values(m_comp["vt"], m_full, PRE_WINDOW)),
    "minimum U(60°N, 10 hPa) (m/s)": (cat_raw.query("group == 'ERA5'")["min_u"].values, cat_raw.query("group == 'E3SM'")["min_u"].values),
    "easterly days": (cat_raw.query("group == 'ERA5'")["easterly_days"].values, cat_raw.query("group == 'E3SM'")["easterly_days"].values),
    "days to recovery": (cat_raw.query("group == 'ERA5'")["days_to_recovery"].dropna().values,
                         cat_raw.query("group == 'E3SM'")["days_to_recovery"].dropna().values),
    "100-hPa polar-cap height, days 15-44 (σ)":
        (event_values(e_comp["pch"], e_full, (15, 44), 100.0), event_values(m_comp["pch"], m_full, (15, 44), 100.0)),
}
fig, axes = plt.subplots(1, len(metrics), figsize=(18, 4.2), gridspec_kw={"wspace": 0.45})
for ax, (name, (eo, mo)) in zip(axes, metrics.items()):
    bp = ax.boxplot([eo[np.isfinite(eo)], mo[np.isfinite(mo)]], patch_artist=True, widths=0.6, whis=(10, 90))
    for patch, c in zip(bp["boxes"], ("0.6", "tab:red")):
        patch.set_facecolor(c)
        patch.set_alpha(0.6)
    ax.set_xticks([1, 2], [f"ERA5\n(n={np.isfinite(eo).sum()})", f"E3SM\n(n={np.isfinite(mo).sum()})"])
    ax.set_title(name, fontsize=fontz * 0.7, loc="left", wrap=True)
    ax.grid(alpha=0.3, axis="y")
fig.suptitle("SSW event properties (unweighted ERA5; whiskers 10-90%)", x=0.12, ha="left", y=1.04)
save(fig, "ssw", "event_properties", YEAR)
plt.show()

rows = []
for label, comp, full, w in (("ERA5, season-matched", e_comp, e_full, e_w), ("E3SM", m_comp, m_full, ones_m)):
    for win in PCH_WINDOWS:
        row = {"set": label, "days": f"{win[0]} to {win[1]}"}
        for p in PCH_LEVELS:
            v = comp["pch"].sel(plev=p, lag=slice(*win)).mean("lag").values[full]
            row[f"Z {p:g} hPa (σ)"] = np.nansum(v * w[full]) / w[full].sum()
        v = comp["slp"].sel(lag=slice(*win)).mean("lag").values[full]
        row["SLP (σ)"] = np.nansum(v * w[full]) / w[full].sum()
        rows.append(row)
vertical = pd.DataFrame(rows)
vertical.to_csv(DATA_DIR / f"ssw_vertical_evolution_{YEAR}.csv", index=False)
print("Polar-cap (60-90°N) anomalies around the SSW central dates (composite means; positive SLP: negative NAM):")
display(vertical.round(2))
